framework

In [11]:
%run universal-framework.ipynb

In [12]:
season_lengths = {
    "std_agri_small": 12,
    "std_agri_medium": 12,
    "std_agri_large": 12,

    "std_energy_small": 24,
    "std_energy_medium": 24,
    "std_energy_large": 24,

    "std_traffic_small": 168,
    "std_traffic_medium": 168,
    "std_traffic_large": 168,

    "std_weather_small": 24,
    "std_weather_medium": 24,
    "std_weather_large": 24,

    "std_exchange_small": 5,
    "std_exchange_medium": 5,
    "std_exchange_large": 5,

    "std_metr_small": 288,
    "std_metr_medium": 288,
    "std_metr_large": 288
}

Forecasting function

In [13]:
def snaive_forecast(
    train,
    horizon,
    season_length
):

    if len(train) < season_length:
        return np.repeat(
            train[-1],
            horizon
        )

    return np.resize(
        train[-season_length:],
        horizon
    )

In [15]:
import gc

results = []

for file in DATA_DIR.glob("*.csv"):

    dataset_name = file.stem

    if dataset_name not in season_lengths:

        print(
            f"Skipping {dataset_name} "
            f"(no season length)"
        )

        continue

    print(f"\nRunning Seasonal Naive on {dataset_name}")

    df = pd.read_csv(
        file,
        low_memory=False
    )

    dataset_horizons = get_horizons_info(
        dataset_name
    )

    for horizon, horizon_type in dataset_horizons:

        print(
            f"  Horizon: {horizon_type}"
            f" ({horizon})"
        )

        season_length = season_lengths[
            dataset_name
        ]

        def forecast_fn(
            train,
            h,
            sl=season_length
        ):
            return snaive_forecast(
                train,
                h,
                sl
            )

        result = benchmark_dataset(
            df=df,
            horizon=horizon,
            forecast_function=forecast_fn,
            model_name="SeasonalNaive"
        )

        if result is not None:

            result["dataset"] = dataset_name
            result["horizon_type"] = horizon_type
            result["season_length"] = season_length

            results.append(result)

            pd.DataFrame(results).to_csv(
                RESULTS_DIR /
                "control_snaive.csv",
                index=False
            )

    del df
    gc.collect()

snaive_results = pd.DataFrame(results)


Running Seasonal Naive on std_agri_large
  Horizon: short (1)
  Horizon: medium (3)
  Horizon: long (5)

Running Seasonal Naive on std_agri_medium
  Horizon: short (1)
  Horizon: medium (3)
  Horizon: long (5)

Running Seasonal Naive on std_agri_small
  Horizon: short (1)
  Horizon: medium (3)
  Horizon: long (5)

Running Seasonal Naive on std_energy_large
  Horizon: short (96)
  Horizon: medium (336)
  Horizon: long (672)

Running Seasonal Naive on std_energy_medium
  Horizon: short (96)
  Horizon: medium (336)
  Horizon: long (672)

Running Seasonal Naive on std_energy_small
  Horizon: short (96)
  Horizon: medium (336)
  Horizon: long (672)
Skipping std_favorita_large_test (no season length)
Skipping std_favorita_large_train (no season length)
Skipping std_favorita_large_transactions (no season length)
Skipping std_favorita_medium_test (no season length)
Skipping std_favorita_medium_train (no season length)
Skipping std_favorita_medium_transactions (no season length)
Skipping std_f

In [16]:
display(
    snaive_results.sort_values(
        ["dataset", "horizon"]
    )
)

,model,n_series,n_rows,horizon,runtime_seconds,MAE,RMSE,SMAPE,dataset,horizon_type,season_length
0,SeasonalNaive,258,15490,1,0.259053,1.412028e+06,1.412028e+06,32.793506,std_agri_large,short,12
1,SeasonalNaive,258,15490,3,0.205230,1.557086e+06,1.597260e+06,32.764444,std_agri_large,medium,12
2,SeasonalNaive,258,15490,5,0.181161,1.483462e+06,1.563662e+06,35.306138,std_agri_large,long,12
3,SeasonalNaive,128,7791,1,0.086233,7.850098e+05,7.850098e+05,35.846833,std_agri_medium,short,12
4,SeasonalNaive,128,7791,3,0.093734,7.213382e+05,7.424222e+05,33.452807,std_agri_medium,medium,12
5,SeasonalNaive,128,7791,5,0.093354,7.275212e+05,7.506899e+05,35.228487,std_agri_medium,long,12
6,SeasonalNaive,27,1560,1,0.021534,9.042317e+05,9.042317e+05,35.586253,std_agri_small,short,12
7,SeasonalNaive,27,1560,3,0.026446,7.357319e+05,7.631880e+05,26.068472,std_agri_small,medium,12
8,SeasonalNaive,27,1560,5,0.024173,7.504159e+05,7.760129e+05,28.729171,std_agri_small,long,12
9,SeasonalNaive,370,51894720,96,5.474754,2.149028e+02,2.616131e+02,44.211531,std_energy_large,short,24


In [17]:
display(
    snaive_results.sort_values("MAE")
)

,model,n_series,n_rows,horizon,runtime_seconds,MAE,RMSE,SMAPE,dataset,horizon_type,season_length
21,SeasonalNaive,207,3547152,12,0.352271,5.209313e+00,6.275102e+00,12.131444,std_traffic_medium,short,168
22,SeasonalNaive,207,3547152,24,0.436590,6.032706e+00,7.379647e+00,14.261554,std_traffic_medium,medium,168
25,SeasonalNaive,207,709389,24,0.217828,7.352441e+00,9.596374e+00,15.907511,std_traffic_small,medium,168
23,SeasonalNaive,207,3547152,48,0.332573,8.352058e+00,1.116460e+01,19.120224,std_traffic_medium,long,168
26,SeasonalNaive,207,709389,48,0.195858,9.058433e+00,1.295150e+01,20.047234,std_traffic_small,long,168
24,SeasonalNaive,207,709389,12,0.269742,1.026804e+01,1.160651e+01,22.834806,std_traffic_small,short,168
20,SeasonalNaive,207,7094304,48,0.508742,1.327601e+01,1.634976e+01,32.517961,std_traffic_large,long,168
18,SeasonalNaive,207,7094304,12,0.583327,1.462588e+01,1.612301e+01,34.945473,std_traffic_large,short,168
19,SeasonalNaive,207,7094304,24,0.505878,1.628885e+01,1.820249e+01,39.987443,std_traffic_large,medium,168
16,SeasonalNaive,370,5189250,336,0.963423,9.440489e+01,1.240543e+02,28.934802,std_energy_small,medium,24
